# 02 · Data preparation

**Owner:** Charlotte

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from src.data import load_train
from src.preprocessing import add_engineered_features, build_preprocessor

import matplotlib.pyplot as plt

FIG_PATH = os.path.join("..", "results", "figures", "eda")
os.makedirs(FIG_PATH, exist_ok=True)

In [ ]:
X_train, y_train = load_train()

preprocessor = build_preprocessor(use_engineered=False)
X_transformed = preprocessor.fit_transform(X_train)
print("Shape without engineered features:", X_transformed.shape)

X_eng = add_engineered_features(X_train)
preprocessor_eng = build_preprocessor(use_engineered=True)
X_transformed_eng = preprocessor_eng.fit_transform(X_eng)
print("Shape with engineered features:", X_transformed_eng.shape)

preprocessor_bins = build_preprocessor(use_engineered=True, use_bins=True)
print("Shape with engineered features + bins:", preprocessor_bins.fit_transform(X_eng).shape)

**Preprocessor Shape Check**\
- Without engineered features: (1684, 35) — 15 continuous (scaled) + 4 one-hot encoded Ethnicity categories + 15 binary + 1 EducationLevel = 35 columns
- With engineered features: (1684, 37) — adds SymptomCount and MotorSymptomCount to the continuous block (+2) = 37 columns
- With engineered features + bins: (1684, 67) — adds 10 one-hot decile bins for each of UPDRS, MoCA and FunctionalAssessment (+30) = 67 columns

In [ ]:
X_eng["Diagnosis"] = y_train.values

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

X_eng.boxplot(column="PulsePressure", by="Diagnosis", ax=axes[0], medianprops={"color": "red"})
axes[0].set_title("PulsePressure\n(EDA only — dropped)")
axes[0].set_xlabel("Diagnosis (0=Non-PD, 1=PD)")

X_eng.boxplot(column="SymptomCount", by="Diagnosis", ax=axes[1], medianprops={"color": "red"})
axes[1].set_title("SymptomCount\n(Non-motor symptoms)")
axes[1].set_xlabel("Diagnosis (0=Non-PD, 1=PD)")

X_eng.boxplot(column="MotorSymptomCount", by="Diagnosis", ax=axes[2], medianprops={"color": "red"})
axes[2].set_title("MotorSymptomCount\n(Cardinal motor symptoms)")
axes[2].set_xlabel("Diagnosis (0=Non-PD, 1=PD)")

plt.suptitle("Engineered Features by Diagnosis", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, "engineered_features.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
X_eng_corr = X_eng.copy()  

engineered_cols = ["PulsePressure", "SymptomCount", "MotorSymptomCount"]
corr_with_diagnosis = X_eng_corr[engineered_cols + ["Diagnosis"]].corr()["Diagnosis"].drop("Diagnosis")

print("--- Engineered Feature Correlations with Diagnosis ---")
print(corr_with_diagnosis.round(3))

print("\n--- Key Original Feature Correlations (for reference) ---")
original_corr = X_eng_corr[["UPDRS", "MoCA", "FunctionalAssessment", "Diagnosis"]].corr()["Diagnosis"].drop("Diagnosis")
print(original_corr.round(3))


## Data preperation justifications:

### Scaling — StandardScaler
Support Vector Machine (SVM) and Multilayer Perceptron (MLP) are both sensitive to feature scale. SVM maximises a margin in feature space — large-scale features would dominate the distance calculation. MLP uses gradient descent — unscaled features cause slow or unstable convergence. StandardScaler (zero mean, unit variance) is applied to all 15 continuous features and the two engineered symptom counts. The scaler is fitted inside each CV fold via Pipeline to prevent data leakage.

### Ethnicity — OneHotEncoder
Ethnicity is a nominal categorical variable with no natural ordering between categories. Assigning integer codes (0, 1, 2...) would imply a false ordinal relationship. OneHotEncoder creates a binary column per category, avoiding this. handle_unknown="ignore" ensures unseen categories during test time do not cause errors.

### Passthrough Features — Binary & EducationLevel

**Binary features (15 columns):**
All binary features (e.g. Gender, Smoking, Tremor, Rigidity...) are already encoded as 0/1 integers. No transformation is needed — they are on a comparable scale by definition and passed through as-is.

**EducationLevel (ordinal):**
EducationLevel has a natural ordering: None < High School < Bachelor's < Higher. The integer codes (0–3) already reflect this progression, so no additional encoding is needed. It is passed through as-is alongside the binary features.

### Imputation — None required
check_quality() in the data understanding notebook confirmed zero missing values across all features. No imputation step is included in the pipeline.

### Outliers — Retained
A range-based check in the data understanding notebook confirmed that all 15 continuous features fall within their documented ranges. Z-score and IQR methods do not apply here — all continuous features follow a uniform distribution (by synthetic data design), for which the maximum possible Z-score is approximately 1.8, making a threshold of Z > 3 impossible to trigger. No observations are removed.

### Engineered Features

**MotorSymptomCount = sum of Tremor, Rigidity, Bradykinesia, PosturalInstability**\
Counts the four cardinal motor signs of Parkinson's Disease. On the train set it reaches r = 0.405 with Diagnosis — marginally above UPDRS (r = 0.402), making them effectively equally strong predictors. Box plots confirm clearly higher counts in PD patients. Computed row-by-row from each patient's own values, so no data leakage is introduced. Serves as a natural continuous FIS input in place of the binary PosturalInstability flag.

**SymptomCount = sum of SpeechProblems, SleepDisorders, Constipation**\
SymptomCount shows very weak correlation with Diagnosis (r = 0.011) and limited separation in box plots — consistent with the symptom frequency chart in notebook 01 which showed near-identical prevalence for these three non-motor symptoms across classes. Despite weak discriminative power individually, it is retained as a non-overlapping complement to 
MotorSymptomCount, capturing a different aspect of symptom burden. Computed row-by-row so no data leakage is introduced.

**PulsePressure = SystolicBP − DiastolicBP**\
PulsePressure is **dropped** from the feature set. Correlation with Diagnosis is negligible (r = 0.011) and box plots show near-identical distributions for PD and Non-PD patients with overlapping medians (~45 mmHg) and interquartile ranges. It adds no meaningful discriminative information beyond what SystolicBP and DiastolicBP already capture individually.

| Feature | r with Diagnosis | Decision |
|---|---|---|
| MotorSymptomCount | +0.405 | Kept — strongest engineered feature |
| UPDRS (reference) | +0.402 | Reference — strongest original feature |
| SymptomCount | +0.011 | Kept — complement to MotorSymptomCount |
| PulsePressure | +0.011 | Dropped — no discriminative value |

### Binned clinical scales
The EDA quintile plots (notebook 01) show that PD risk changes in steps along UPDRS, MoCA and FunctionalAssessment rather than linearly. The plot below shows the PD rate per decile of each scale on the training set.

In [ ]:
import pandas as pd

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, col in zip(axes, ["UPDRS", "MoCA", "FunctionalAssessment"]):
    decile = pd.qcut(X_train[col], 10, labels=False) + 1   # 1 = lowest 10%, 10 = highest 10%
    y_train.groupby(decile).mean().plot(marker="o", ax=ax, label="PD rate in decile")
    ax.axhline(y_train.mean(), color="grey", ls=":", label="overall PD rate")
    ax.set_title(f"PD rate per {col} decile"); ax.set_xlabel(f"{col} decile (1 = lowest, 10 = highest)")
    ax.set_xticks(range(1, 11)); ax.set_ylabel("PD rate"); ax.legend()
plt.suptitle("Step-like relationships motivating the binned features (train set)", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, "binned_features.png"), dpi=150, bbox_inches="tight")
plt.show()

To make these threshold-like patterns easier to learn for the SVM and MLP, each of the three scales is additionally split into 10 quantile bins (deciles) and one-hot encoded (`build_preprocessor(use_engineered=True, use_bins=True)`). The scaled original values stay in, so no information is lost, and no feature selection is applied.

**Leakage prevention:** the binning is a step of the scikit-learn `Pipeline`, like the scaler. In every cross-validation fold the pipeline is fitted on the training folds only, so the decile edges are computed from those patients alone and then applied unchanged to the held-out fold. The final model learns its edges from the full training set, and the test set is binned with these fixed edges in `04_evaluation`. The binning is unsupervised (it never uses the diagnosis). The FIS does not use the bins.